In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copy import deepcopy
from bertopic import BERTopic
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration
from transformers import pipeline

In [5]:
# Load data from Hugging Face
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

# Extract metadata
abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

# Embedding Documents

In [8]:
# Create an embedding for each abstract
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


ValueError: Unsupported input type: Column. Expected one of: str, dict, PIL.Image.Image, np.ndarray, torch.Tensor

In [ ]:
embeddings.shape

In [ ]:
len(embeddings[0])

# Reducing the Dimensionality of Embeddings
Well-known methods for dimensionality reduction are Principal Component Analysis (PCA) and Uniform Manifold Approximation and Projection (UMAP).


In [ ]:
# We reduce the input embeddings from 384 dimensions to 5 dimensions
umap_model = UMAP(n_components=5, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_model.fit_transform(embeddings)

# Cluster the Reduced Embeddings

In [ ]:
# We fit the model and extract the clusters
hdbscan_model = HDBSCAN(min_cluster_size=50, metric="euclidean", cluster_selection_method="eom").fit(reduced_embeddings)

clusters = hdbscan_model.labels_

In [ ]:
# How many clusters did we generate?
len(set(clusters))

# Inspecting the Clusters

In [ ]:
# Print first three documents in cluster 0
cluster = 0
for index in np.where(clusters==cluster)[0][:3]:
    print(abstracts[index][:300] + "... \n")

In [ ]:
# Reduce 384-dimensional embeddings to two dimensions for easier visualization
reduced_embeddings = UMAP(
 n_components=2, min_dist=0.0, metric="cosine", random_state=42
).fit_transform(embeddings)

# Create dataframe
df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
df["title"] = titles
df["cluster"] = [str(c) for c in clusters]

# Select outliers and non-outliers (clusters)
to_plot = df.loc[df.cluster != "-1", :]
outliers = df.loc[df.cluster == "-1", :]


In [ ]:
clusters_df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
clusters_df["title"] = titles
clusters_df["cluster"] = [str(c) for c in clusters]

outliers_df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
outliers_df["title"] = titles
outliers_df["cluster"] = [str(c) for c in clusters]

In [ ]:
# Plot outliers and non-outliers separately
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="grey")
plt.scatter(
 clusters_df.x, clusters_df.y, c=clusters_df.cluster.astype(int),
 alpha=0.6, s=2, cmap="tab20b"
)
plt.axis("off")

# From Text Clustering to Topic Modeling
This idea of finding themes or latent topics in a collection of textual data is often referred to as topic modeling. 

BERTopic: A Modular Topic Modeling Framework

BERTopic is a topic modeling technique that leverages clusters of semantically similar texts to extract various types of topic representations.6

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
).fit(abstracts, embeddings)

In [ ]:
topic_model.get_topic_info()

For example, topic 0 contains the keywords “speech,” “asr,” and “recognition.” Based
on these keywords, it seems that the topic is about automatic speech recognition
(ASR)

In [ ]:
topic_model.get_topic(0)

In [ ]:
topic_model.find_topics("topic modeling")

In [ ]:
topic_model.get_topic(22)

In [ ]:
# Visualize topics and documents
fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1200,
    hide_annotations=True
)

# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [ ]:
# Visualize barchart with ranked keywords
topic_model.visualize_barchart()

In [ ]:
# Visualize relationships between topics
topic_model.visualize_heatmap(n_clusters=30)

In [ ]:
# Visualize the potential hierarchical structure of topics
topic_model.visualize_hierarchy()

# Adding a Special Lego Block
The pipeline in BERTopic that we have explored thus far, albeit fast and modular, has a disadvantage: it still represents a topic through a bag-of-words without taking into account semantic structures.

The solution is to leverage the strength of the bag-of-words representation, which is its speed to generate a meaningful representation.

In [ ]:
# Save original representations
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    
    for topic in range(nr_topics):
        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]
 
    return df

# KeyBERTInspired
The first representation block that we are going to explore is KeyBERTInspired. KeyBERTInspired is, as you might have guessed, a method inspired by the keyword extraction package, KeyBERT. KeyBERT extracts keywords from texts by comparing word and document embeddings through cosine similarity

In [ ]:
# Update our topic representations using KeyBERTInspired
representation_model = KeyBERTInspired()
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

# Maximal marginal relevance
We can use maximal marginal relevance (MMR) to diversify our topic representations. The algorithm attempts to find a set of keywords that are diverse from one another but still relate to the documents they are compared to. 

In [ ]:
# Update our topic representations to MaximalMarginalRelevance
representation_model = MaximalMarginalRelevance(diversity=0.2)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

# The Text Generation Lego Block

In [ ]:
prompt = """I have a topic that contains the following documents: 
[DOCUMENTS]
The topic is described by the following keywords: '[KEYWORDS]'.
Based on the documents and keywords, what is this topic about?"""

In [ ]:
# Update our topic representations using Flan-T5
generator = pipeline("text2text-generation", model="google/flan-t5-small")
representation_model = TextGeneration(generator, prompt=prompt, doc_length=50, tokenizer="whitespace")
topic_model.update_topics(abstracts, representation_model=representation_model)

In [ ]:
# Show topic differences
topic_differences(topic_model, original_topics)

In [ ]:
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]
Based on the information above, extract a short topic label in the following 
format:
topic: <short topic label>
"""


# yout can try gpt GPT-3.5. you need to see page 185